In [1]:
# Differentiable Logic Gate Network
# Implementacja zgodna z artykułem "Deep Differentiable Logic Gate Networks"
# Autor implementacji: asystent (przykładowy kod do uruchomienia lokalnie)

import math
import random
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

# ----------------------
# Real-valued logic ops
# ----------------------
# Zaimplementowane zgodnie z Table 1 w artykule (probabilistic interpretation)
# Każda funkcja przyjmuje dwa tensory a, b (dowolny shape) z wartościami w [0,1]

def op_false(a, b):
    return torch.zeros_like(a)

def op_and(a, b):
    return a * b

def op_not_a_implies_b(a, b):
    # ¬(A ⇒ B) = A - A*B
    return a - a * b

def op_a(a, b):
    return a

def op_not_b_implies_a(a, b):
    # ¬(A ⇐ B) = B - A*B
    return b - a * b

def op_b(a, b):
    return b

def op_xor(a, b):
    return a + b - 2 * a * b

def op_or(a, b):
    return a + b - a * b

def op_not_or(a, b):
    return 1.0 - (a + b - a * b)

def op_not_xnor(a, b):
    return 1.0 - (a + b - 2 * a * b)

def op_not_b(a, b):
    return 1.0 - b

def op_b_impl_a(a, b):
    return 1.0 - b + a * b

def op_not_a(a, b):
    return 1.0 - a

def op_a_impl_b(a, b):
    return 1.0 - a + a * b

def op_not_and(a, b):
    return 1.0 - a * b

def op_true(a, b):
    return torch.ones_like(a)

# Lista operatorów (w tej dokładnej kolejności odpowiadającej ID 0..15 z tabeli)
OP_FUNCS = [
    op_false,
    op_and,
    op_not_a_implies_b,
    op_a,
    op_not_b_implies_a,
    op_b,
    op_xor,
    op_or,
    op_not_or,
    op_not_xnor,
    op_not_b,
    op_b_impl_a,
    op_not_a,
    op_a_impl_b,
    op_not_and,
    op_true,
]

# ----------------------
# Differentiable Logic Layer
# ----------------------
class DifferentiableLogicLayer(nn.Module):
    """Warstwa logiki: każdy neuron wybiera (softmax) jednego z 16 operatorów.
    Każdy neuron ma dwa wejścia wybrane pseudolosowo z poprzedniej warstwy.
    Po dyskretyzacji (tryb inference_discrete=True) wybieramy najbardziej prawdopodobny operator.
    """

    def __init__(self, in_size: int, num_neurons: int, seed: int = 0):
        super().__init__()
        self.in_size = in_size
        self.num_neurons = num_neurons
        rng = random.Random(seed)

        # Połączenia: dla każdego neuronu wybieramy dwa indeksy z zakresu [0, in_size)
        # Zapisujemy je jako tensor long (2 x num_neurons)
        conns = []
        for _ in range(num_neurons):
            i1 = rng.randrange(in_size)
            i2 = rng.randrange(in_size)
            conns.append((i1, i2))
        self.register_buffer("conns", torch.tensor(conns, dtype=torch.long))

        # Parametry logistyczne (logits) dla rozkładu kategorii 16 dla każdego neuronu
        # shape: (num_neurons, 16)
        # Zainicjalizowane N(0,1) zgodnie z artykułem
        w = torch.randn(num_neurons, 16) * 1.0
        self.logits = nn.Parameter(w)

    def forward(self, x: torch.Tensor, inference_discrete: bool = False) -> torch.Tensor:
        """x: tensor shape (batch, in_size) lub (batch, in_size, ...) - ale dla nas (batch, in_size)
        Zwraca: tensor shape (batch, num_neurons)
        """
        batch = x.shape[0]
        device = x.device

        # Wybierz inputy per neuron
        # conns: (num_neurons, 2)
        idx = self.conns.to(device)
        a1 = x[:, idx[:, 0]]  # (batch, num_neurons)
        a2 = x[:, idx[:, 1]]  # (batch, num_neurons)

        # Oblicz wszystkie 16 operatorów: uzyskamy tensor shape (batch, num_neurons, 16)
        vals = []
        for f in OP_FUNCS:
            vals.append(f(a1, a2).unsqueeze(-1))
        vals = torch.cat(vals, dim=-1)  # (batch, num_neurons, 16)

        if inference_discrete:
            # wybierz najbardziej prawdopodobny operator per neuron (argmax logits)
            choices = torch.argmax(self.logits, dim=1)  # (num_neurons,)
            # zindeksuj vals po ostatniej osi
            # przygotuj indeksy batch x neurons
            bidx = torch.arange(batch, device=device)[:, None]
            nidx = torch.arange(self.num_neurons, device=device)[None, :]
            chosen = vals[bidx, nidx, choices[None, :].expand(batch, -1)]
            return chosen
        else:
            # użyj softmaxu nad logits -> probabilistyczna mieszanka operatorów
            p = F.softmax(self.logits, dim=-1)  # (num_neurons, 16)
            p = p.unsqueeze(0).expand(batch, -1, -1)  # (batch, num_neurons, 16)
            out = (p * vals).sum(dim=-1)  # (batch, num_neurons)
            return out


# ----------------------
# Differentiable Logic Net
# ----------------------
class DifferentiableLogicNet(nn.Module):
    def __init__(self, input_dim: int, layers: List[int], outputs: int = 1, seed: int = 0):
        super().__init__()
        self.layers_sizes = layers
        self.outputs = outputs
        self.seed = seed

        modules = []
        in_size = input_dim
        for i, n in enumerate(layers):
            modules.append(DifferentiableLogicLayer(in_size, n, seed=seed + i))
            in_size = n
        self.layers = nn.ModuleList(modules)

        # final projection: produce a number of bits / output neurons = outputs
        # jeśli ostatnia warstwa ma więcej neuronów niż outputs, agregujemy ich sumy
        # Użyjemy prostego podejścia: jeśli chcemy predict scalarną wartość, ustaw outputs=K
        # i skaluj przez tau/beta w trainingu. Tutaj outputs określa liczbę "bitów"u wyjściowych.

    def forward(self, x: torch.Tensor, inference_discrete: bool = False) -> torch.Tensor:
        # x: (batch, input_dim)
        h = x
        for layer in self.layers:
            h = layer(h, inference_discrete=inference_discrete)
            # wartości są już w [0,1]
        # h: (batch, last_layer_size)
        # Zwracamy raw outputs (prawdopodobności) - dalsza agregacja po stronie treningu/testu
        return h

    def discretize(self):
        # pomocniczo: ustawia wszystkie warstwy w tryb dyskretny podczas forward
        # (nic nie trzeba modyfikować w parametrze - forward obsługuje inference_discrete)
        pass

# ----------------------
# Helper: prosta sieć MLP dla porównania
# ----------------------
class SimpleMLP(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 64, hidden_layers: int = 2, output_dim: int = 1):
        super().__init__()
        layers = []
        in_d = input_dim
        for i in range(hidden_layers):
            layers.append(nn.Linear(in_d, hidden))
            layers.append(nn.ReLU())
            in_d = hidden
        layers.append(nn.Linear(in_d, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# ----------------------
# Dataset generation (użytkownik dostarczył fragment; tu go integrujemy)
# ----------------------

def custom_round(val):
    return int(val + 0.5)


def make_dataset(size: int) -> Tuple[np.ndarray, np.ndarray]:
    x = np.arange(0, size, 0.1)
    xg, yg = np.meshgrid(x, x)
    dataX = xg.flatten()
    dataY = yg.flatten()
    dataXY = np.column_stack((dataX, dataY))
    data_labels = np.array([custom_round(a) * custom_round(b) for a, b in dataXY], dtype=np.float32)
    return dataXY.astype(np.float32), data_labels

# ----------------------
# Training & evaluation utilities
# ----------------------

def train_logic_net(X_train, y_train, X_test, y_test, device='cpu', 
                    layers=[32, 32], outputs_per_example=1, epochs=100, lr=0.01, tau=1.0):
    input_dim = X_train.shape[1]
    model = DifferentiableLogicNet(input_dim, layers, outputs=outputs_per_example, seed=42).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    Xtr = torch.from_numpy(X_train).to(device)
    ytr = torch.from_numpy(y_train).unsqueeze(-1).to(device)
    Xte = torch.from_numpy(X_test).to(device)
    yte = torch.from_numpy(y_test).unsqueeze(-1).to(device)

    batch_size = min(256, Xtr.shape[0])
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(Xtr.shape[0], device=device)
        losses = []
        for i in range(0, Xtr.shape[0], batch_size):
            idx = perm[i:i+batch_size]
            xb = Xtr[idx]
            yb = ytr[idx]
            optimizer.zero_grad()
            out_bits = model(xb, inference_discrete=False)  # (batch, last_layer_size)
            # agregujemy: uśredniamy wartości i mapujemy liniowo z tau
            # W prostym ustawieniu weźmiemy mean po osi neuronów -> pred
            pred = out_bits.mean(dim=1, keepdim=True) / tau
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if (ep + 1) % max(1, epochs // 5) == 0:
            # walidacja
            model.eval()
            with torch.no_grad():
                out_tr = model(Xtr, inference_discrete=False).mean(dim=1, keepdim=True) / tau
                out_te = model(Xte, inference_discrete=False).mean(dim=1, keepdim=True) / tau
                train_mse = loss_fn(out_tr, ytr).item()
                test_mse = loss_fn(out_te, yte).item()
            print(f"Epoch {ep+1}/{epochs} train_loss={np.mean(losses):.4f} train_mse={train_mse:.4f} test_mse={test_mse:.4f}")

    # Dyskretyzacja i finalna ocena
    model.eval()
    with torch.no_grad():
        out_te = model(Xte, inference_discrete=True).mean(dim=1, keepdim=True) / tau
        test_mse_disc = loss_fn(out_te, yte).item()
    print(f"Final (discrete inference) test MSE: {test_mse_disc:.4f}")
    return model, test_mse_disc


def train_mlp(X_train, y_train, X_test, y_test, device='cpu', hidden=64, epochs=100, lr=1e-3):
    input_dim = X_train.shape[1]
    model = SimpleMLP(input_dim, hidden=hidden, hidden_layers=2, output_dim=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    Xtr = torch.from_numpy(X_train).to(device)
    ytr = torch.from_numpy(y_train).unsqueeze(-1).to(device)
    Xte = torch.from_numpy(X_test).to(device)
    yte = torch.from_numpy(y_test).unsqueeze(-1).to(device)

    batch_size = min(256, Xtr.shape[0])
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(Xtr.shape[0], device=device)
        losses = []
        for i in range(0, Xtr.shape[0], batch_size):
            idx = perm[i:i+batch_size]
            xb = Xtr[idx]
            yb = ytr[idx]
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if (ep + 1) % max(1, epochs // 5) == 0:
            model.eval()
            with torch.no_grad():
                train_mse = loss_fn(model(Xtr), ytr).item()
                test_mse = loss_fn(model(Xte), yte).item()
            print(f"MLP Epoch {ep+1}/{epochs} train_loss={np.mean(losses):.4f} train_mse={train_mse:.4f} test_mse={test_mse:.4f}")
    model.eval()
    with torch.no_grad():
        test_mse = loss_fn(model(Xte), yte).item()
    print(f"MLP final test MSE: {test_mse:.4f}")
    return model, test_mse

# ----------------------
# Demo: trenujemy na trzech rozmiarach (jak w Twoim snippet'cie)
# ----------------------
if __name__ == '__main__':
    import time
    device = 'cpu'

    sizes = [2, 3, 4]
    results = []
    for size in sizes:
        print('\n' + '='*60)
        print(f'Dataset size: {size}x{size}')
        X, y = make_dataset(size)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)

        # Normalizacja prostą skalą (opcjonalne)
        # Skaluje wejścia do [0,1]
        max_input = np.max(X)
        X_train_s = X_train / max_input
        X_test_s = X_test / max_input

        # Logic Net - małe ustawienia do demonstracji (szybkie)
        print('\nTraining Differentiable Logic Net:')
        start = time.time()
        logic_model, logic_mse = train_logic_net(X_train_s, y_train, X_test_s, y_test,
                                                 device=device, layers=[64, 64], epochs=100, lr=0.01, tau=1.0)
        took = time.time() - start
        print(f'Logic net finished in {took:.1f}s test_mse={logic_mse:.4f}')

        # Porównanie: prosty MLP
        print('\nTraining MLP baseline:')
        start = time.time()
        mlp_model, mlp_mse = train_mlp(X_train_s, y_train, X_test_s, y_test, device=device, hidden=64, epochs=100, lr=1e-3)
        took2 = time.time() - start
        print(f'MLP finished in {took2:.1f}s test_mse={mlp_mse:.4f}')

        results.append((size, logic_mse, mlp_mse))

    print('\nSummary results (size, logic_mse, mlp_mse):')
    for r in results:
        print(r)

    print('\nUwaga: To jest referencyjna i edukacyjna implementacja. Artykuł zawiera
szereg optymalizacji (np. natywne CUDA, wektoryzacja bitowa, agregatory binarne),
które nie zostały tutaj zaimplementowane. Ten kod ma służyć do eksperymentów
na małych danych i weryfikacji koncepcji.')


SyntaxError: unterminated string literal (detected at line 362) (1237885255.py, line 362)